In [12]:
from pymilvus import MilvusClient, DataType, Function, FunctionType
from app.core.config import settings

client = MilvusClient(uri = settings.rag.milvus_url)

#创建字段结构field (column)，关闭自动ID功能，取消动态补足字段
schema = client.create_schema(auto_id = False,enable_dynamic_field = False)
schema.add_field(field_name="id",datatype=DataType.INT64,is_primary = True)
schema.add_field(field_name="h1",datatype=DataType.VARCHAR,max_length = 128)
schema.add_field(field_name="h2",datatype=DataType.VARCHAR,max_length = 128)
schema.add_field(field_name="h3",datatype=DataType.VARCHAR,max_length = 128)
schema.add_field(
     field_name="text",         #字段名称
     datatype=DataType.VARCHAR, #字段类型
     max_length = 2048,         #字段长度
     enable_analyzer = True,    #允许分词，主要是生成稀疏向量时需要从这里提取文本片段中的关键词
     analyzer_params = {"type":"chinese"}#采用中文分词方式，与英文不同
)
schema.add_field(field_name="dense",datatype=DataType.FLOAT_VECTOR,dim = 1024)
schema.add_field(field_name="sparse",datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_function(Function(
    name="text-bm25-emb",               #函数名称，可自定义
    function_type = FunctionType.BM25,  #函数是BM25统计式算法
    input_field_names="text",           #让Milvus从text字段读取文本，再使用BM25计算稀疏向量
    output_field_names="sparse",        #计算后的稀疏向量存储到sparse字段里
))

#添加实体结构entity (row)
index_params = client.prepare_index_params()
index_params.add_index(field_name="h2",index_type="AUTOINDEX")
index_params.add_index(field_name="dense",index_type="AUTOINDEX",metric_type = "COSINE")
index_params.add_index(field_name="sparse",index_type="SPARSE_INVERTED_INDEX",metric_type = "BM25")
client.create_collection(
    collection_name="my_1",
    schema=schema,
    index_params = index_params
)
#client.drop_collection(collection_name="mycolection1")